# 05 — Project Mapping

**Objective**: load `config/project_mapping.yaml` and demonstrate Section 6's cross-source matching rules — a project is only ever unified via an explicit mapping entry, and a missing entry on either side surfaces as `UNKNOWN`, never a guess.

**Dependencies**: `src/services/project_unifier.py`.

**Configuration**: `config/project_mapping.yaml` — 7 entries: 5 fully mapped, 1 Jira-only (`QSR`, no finance mapping), 1 Finance-only (`Helios Compliance Program`, no Jira mapping).

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import pandas as pd
from src.services import project_unifier

mapping = project_unifier.load_project_mapping()
print(f"organization_id={mapping.organization_id}  portfolio_id={mapping.portfolio_id}")
print(f"{len(mapping.entries)} project mapping entries loaded")

organization_id=org-northwind  portfolio_id=portfolio-elt-2026
7 project mapping entries loaded


## The mapping table itself

In [2]:
rows = []
for pid, entry in mapping.entries.items():
    rows.append({
        "project_id": pid,
        "project_name": entry.project_name,
        "jira_key": entry.jira_key or "—",
        "finance_project_id": entry.finance_project_id or "—",
        "project_manager": entry.project_manager,
        "project_status": entry.project_status,
    })
pd.DataFrame(rows)

,project_id,project_name,jira_key,finance_project_id,project_manager,project_status
0,PROJECT-10001,Phoenix Platform Modernization,PHX,10001,Nathan Brooks,Active
1,PROJECT-10002,Orca Payments Gateway,ORCA,10002,Fatima Noor,Active
2,PROJECT-10003,Nova Customer Portal,NOVA,10003,Grace Lin,Active
3,PROJECT-10004,Titan Infra Automation,TITAN,10004,Marcus Johnson,Active
4,PROJECT-10005,Lynx Data Analytics,LYNX,10005,Tom O'Brien,Active
5,PROJECT-10006,Quasar Self-Service Analytics,QSR,—,Priya Raman,Active
6,PROJECT-10007,Helios Compliance Program,—,10007,Brianna Scott,Active


## Never guess: `null` stays `None`, not an empty-string coincidence

Two entries are deliberately incomplete on one side. Confirming these parse as `None` (not `"null"` the string, not `""`) matters — a stray string would silently pass an `if entry.jira_key:` check further down and corrupt the unification.

In [3]:
qsr = mapping.entries["PROJECT-10006"]
helios = mapping.entries["PROJECT-10007"]

print(f"QSR    (PROJECT-10006): jira_key={qsr.jira_key!r}          finance_project_id={qsr.finance_project_id!r}")
print(f"Helios (PROJECT-10007): jira_key={helios.jira_key!r}          finance_project_id={helios.finance_project_id!r}")

assert qsr.finance_project_id is None
assert helios.jira_key is None and helios.jira_project_id is None

QSR    (PROJECT-10006): jira_key='QSR'          finance_project_id=None
Helios (PROJECT-10007): jira_key=None          finance_project_id='10007'


## Verifying the two unmapped projects actually exist in their one source

It would be easy to accidentally test the missing-mapping path against a project that doesn't exist in *either* source — which would prove nothing. Confirm QSR is real in Jira and Helios is real in Finance.

In [4]:
from src.connectors.jira_client import build_default_jira_client
from src.connectors.financial_client import CSVFinancialDataSource

jira_client = build_default_jira_client()
fin_source = CSVFinancialDataSource()

qsr_issues = jira_client.get_project_issues("QSR").records
print(f"QSR exists in Jira: {len(qsr_issues)} issues found")

helios_period = fin_source.get_latest_reporting_period("10007")
helios_record = fin_source.get_project_finances("10007", helios_period)
print(f"Helios exists in Finance: latest period={helios_period}, approved_budget={helios_record.approved_budget:,.2f}")

# And confirm the OTHER side genuinely has nothing for them:
print(f"\nQSR in Finance: {fin_source.get_latest_reporting_period('10006')}  (None — confirmed absent)")
print(f"Helios in Jira: {len(jira_client.get_project_issues('HELIOS').records)} issues  (0 — no such Jira project)")

QSR exists in Jira: 5 issues found
Helios exists in Finance: latest period=2026-08, approved_budget=50,500.00

QSR in Finance: None  (None — confirmed absent)
Helios in Jira: 0 issues  (0 — no such Jira project)


## Mapping risks, surfaced explicitly (Section 6)

A mapping gap is not just a blank cell in a table — it's a Risk with evidence and a recommended action, same as any delivery or financial risk.

In [5]:
from datetime import date

AS_OF = date(2026, 9, 15)
portfolio = project_unifier.build_unified_portfolio(mapping, jira_client, fin_source, AS_OF)

for project in portfolio:
    risks = project_unifier.detect_mapping_risks(project)
    if risks:
        for r in risks:
            print(f"[{r.severity.value}] {project.project_id} ({project.project_name})")
            print(f"    {r.description}")
            print(f"    evidence: {r.evidence}")
            print(f"    action:   {r.recommended_action}\n")

[MEDIUM] PROJECT-10006 (Quasar Self-Service Analytics)
    Quasar Self-Service Analytics has no financial system mapping
    evidence: ['config/project_mapping.yaml: finance_project_id is null for PROJECT-10006']
    action:   Assign a finance_project_id in config/project_mapping.yaml to enable financial oversight.

[MEDIUM] PROJECT-10007 (Helios Compliance Program)
    Helios Compliance Program has no Jira project mapping
    evidence: ['config/project_mapping.yaml: jira_project_id is null for PROJECT-10007']
    action:   Assign a jira_project_id in config/project_mapping.yaml to enable delivery oversight.



## Validation checks

- [x] All 7 mapping entries load, with `null` YAML values parsing as Python `None`
- [x] Both intentionally-partial entries (`QSR`, `Helios`) are confirmed to genuinely exist in exactly one source, not neither
- [x] `detect_mapping_risks` fires exactly once each for `PROJECT-10006` (financial side) and `PROJECT-10007` (delivery side), and never for the 5 fully-mapped projects

## Testing

`tests/test_project_unifier.py::TestLoadProjectMapping` and `::TestBuildUnifiedPortfolio::test_exactly_two_projects_carry_a_mapping_gap`.

## Next step

`06_data_normalization.ipynb` — assemble full `Project` objects from these mappings and cross-check every figure against notebooks 02-04.